# Validation du modèle — Identification de fragments parallèles

Ce notebook applique les modèles de régression logistique entraînés précédemment à un corpus comparable non aligné composé de textes bruts en français et en langues régionales (alsacien, corse, occitan).

**Structure attendue du dossier `validation/` :**
```
validation/
├── alsacien_gsw/
│   └── fr-gsw/
│       ├── article_fr.txt
│       └── article_gsw.txt
├── corse_co/
│   └── fr-co/
│       ├── article_fr.txt
│       └── article_co.txt
└── occitan_oc/
    └── fr-oc/
        ├── article_fr.txt
        └── article_oc.txt
```

**Pipeline :**
1. Chargement et nettoyage des textes bruts
2. Segmentation en phrases
3. Calcul des variables d'apprentissage
4. Inférence avec le modèle binaire et multiclasse
5. Analyse des résultats par langue
6. Export CSV des paires candidates

## 0. Imports et configuration

In [ ]:
import os
import re
import unicodedata
import string
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import jellyfish
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jellyfish", "-q"])
    import jellyfish

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim
    _sbert_available = True
except ImportError:
    _sbert_available = False
    print("[AVERTISSEMENT] sentence-transformers non disponible — score SBERT mis à 0.")

# -------------------------------------------------------
# Configuration — à adapter si nécessaire
# -------------------------------------------------------
VALIDATION_DIR   = "validation/"          # dossier racine des données de validation
MODEL_BIN_PATH   = "logreg_binary.joblib"
MODEL_MC_PATH    = "logreg_multiclass.joblib"
SCALER_PATH      = "scaler.joblib"
SBERT_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
OUTPUT_CSV       = "validation_candidates.csv"

# Seuil de confiance minimum pour retenir une paire candidate
CONFIDENCE_THRESHOLD = 0.65

# Correspondance dossier → code langue
LANG_FOLDERS = {
    "alsacien_gsw": "gsw",
    "corse_co":     "co",
    "occitan_oc":   "oc",
}

FEATURE_COLS = [
    "sbert_score",
    "length_score",
    "jaccard_bigrams",
    "jaccard_trigrams",
    "jaccard_quadrigrams",
    "phonetic_jaccard",
]

print("Configuration chargée.")

## 1. Chargement des modèles

In [ ]:
scaler     = joblib.load(SCALER_PATH)
model_bin  = joblib.load(MODEL_BIN_PATH)
model_mc   = joblib.load(MODEL_MC_PATH)
print(f"  ✔ Scaler chargé          : {SCALER_PATH}")
print(f"  ✔ Modèle binaire chargé  : {MODEL_BIN_PATH}")
print(f"  ✔ Modèle multiclasse chargé : {MODEL_MC_PATH}")

if _sbert_available:
    sbert_model = SentenceTransformer(SBERT_MODEL_NAME)
    print(f"  ✔ Modèle SBERT chargé    : {SBERT_MODEL_NAME}")

# Récupère les étiquettes de classe du modèle multiclasse
mc_classes = model_mc.classes_
print(f"  Classes multiclasse : {mc_classes}")

## 2. Nettoyage et segmentation des textes bruts

Les fichiers sont des textes bruts continus (paragraphes, listes, titres de section). 
Le pipeline de nettoyage effectue les opérations suivantes dans l'ordre :

1. **Suppression des lignes non informatives** : lignes vides, titres de section courts, éléments de listes isolés (lignes commençant par une majuscule isolée ou un tiret sans verbe).
2. **Normalisation des espaces et de la ponctuation** : suppression des espaces multiples, normalisation des tirets et guillemets.
3. **Segmentation en phrases** : découpage sur les ponctuations fortes (`.`, `!`, `?`) avec gestion des abréviations courantes et des nombres.
4. **Filtrage des phrases trop courtes** : suppression des phrases de moins de 5 tokens, qui ne portent pas assez d'information sémantique pour être comparées.

In [ ]:
def clean_text(raw: str) -> str:
    """
    Nettoie un texte brut avant segmentation.
    Supprime les éléments de liste isolés, les titres courts,
    normalise les espaces et la ponctuation.
    """
    lines = raw.splitlines()
    cleaned = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Supprime les lignes-titres courtes (≤ 4 mots, pas de verbe apparent)
        words = line.split()
        if len(words) <= 4 and not re.search(r'[.!?]$', line):
            continue
        # Supprime les lignes qui sont de simples noms propres ou éléments de liste
        if re.match(r'^[A-ZÁÀÂÉÈÊËÎÏÔÙÛÜÇ][a-záàâéèêëîïôùûüç]+(\s[A-ZÁÀÂÉÈÊËÎÏÔÙÛÜÇ][a-z]+)*$', line) and len(words) <= 3:
            continue
        cleaned.append(line)
    text = " ".join(cleaned)
    # Normalisation des espaces
    text = re.sub(r'\s+', ' ', text)
    # Normalisation des tirets longs
    text = re.sub(r'[–—]', '-', text)
    return text.strip()


def segment_sentences(text: str, min_tokens: int = 5) -> list:
    """
    Segmente un texte nettoyé en phrases.
    Gère les abréviations courantes et les nombres décimaux
    pour éviter les coupures erronées.
    Filtre les phrases de moins de `min_tokens` mots.
    """
    # Protège les abréviations courantes avant la segmentation
    abbreviations = [
        r'M\.', r'Mme\.', r'Dr\.', r'St\.', r'etc\.', r'cf\.', r'p\.ex\.', r'art\.',
        r'vol\.', r'n°', r'fig\.', r'env\.', r'av\.', r'ap\.',
    ]
    placeholders = {}
    for i, abbr in enumerate(abbreviations):
        placeholder = f"__ABBR{i}__"
        text, n = re.subn(abbr, placeholder, text)
        if n:
            placeholders[placeholder] = re.sub(r'\\', '', abbr)

    # Protège les nombres décimaux (ex: 2 706, 1.5)
    text = re.sub(r'(\d)\.(\d)', r'\1__DOT__\2', text)

    # Segmentation sur . ! ? suivis d'une majuscule ou fin de chaîne
    sentences_raw = re.split(r'(?<=[.!?])\s+(?=[A-ZÁÀÂÉÈÊËÎÏÔÙÛÜÇА-ЯA-Z"\'«])', text)

    sentences = []
    for sent in sentences_raw:
        # Restaure les placeholders
        for ph, original in placeholders.items():
            sent = sent.replace(ph, original)
        sent = sent.replace('__DOT__', '.')
        sent = sent.strip()
        if len(sent.split()) >= min_tokens:
            sentences.append(sent)

    return sentences


def load_and_segment(filepath: str) -> list:
    """
    Lit un fichier texte brut, le nettoie et le segmente en phrases.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()
    cleaned = clean_text(raw)
    return segment_sentences(cleaned)


print("Fonctions de nettoyage et segmentation définies.")

In [ ]:
# -------------------------------------------------------
# Vérification sur les fichiers d'exemple joints
# (Bacourt.txt et Corsica.txt si disponibles)
# -------------------------------------------------------
for test_file in ["Bacourt.txt", "Corsica.txt"]:
    if os.path.exists(test_file):
        sents = load_and_segment(test_file)
        print(f"\n[{test_file}] → {len(sents)} phrases extraites")
        for i, s in enumerate(sents[:4], 1):
            print(f"  {i}. {s[:90]}")
    else:
        print(f"[{test_file}] non trouvé dans le répertoire courant — ignoré.")

## 3. Calcul des variables d'apprentissage

Les fonctions de calcul des variables sont identiques à celles utilisées lors de la constitution du jeu de données d'entraînement.

In [ ]:
def normalize(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text.translate(str.maketrans("", "", string.punctuation)).strip()

def length_score(s1: str, s2: str) -> float:
    n1, n2  = len(s1.split()), len(s2.split())
    max_len = max(n1, n2)
    return 0.0 if max_len == 0 else abs(n1 - n2) / max_len

def char_ngrams(text: str, n: int) -> set:
    t = normalize(text)
    return set(t[i:i+n] for i in range(len(t) - n + 1))

def jaccard_index(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def jaccard_ngrams(s1: str, s2: str, n: int) -> float:
    return jaccard_index(char_ngrams(s1, n), char_ngrams(s2, n))

def metaphone_codes(text: str) -> set:
    codes = set()
    for word in normalize(text).split():
        try:
            code = jellyfish.metaphone(word)
            if code:
                codes.add(code)
        except Exception:
            pass
    return codes

def phonetic_jaccard(s1: str, s2: str) -> float:
    return jaccard_index(metaphone_codes(s1), metaphone_codes(s2))


def compute_features_batch(fr_sentences: list, lr_sentences: list,
                            sbert_model=None, batch_size: int = 64) -> np.ndarray:
    """
    Calcule le vecteur de variables pour une liste de paires (fr, lr).
    Retourne un array de forme (N, 6).
    """
    n = len(fr_sentences)

    # Score SBERT en batch
    sbert_scores = np.zeros(n)
    if sbert_model is not None:
        for i in range(0, n, batch_size):
            b_fr = fr_sentences[i:i+batch_size]
            b_lr = lr_sentences[i:i+batch_size]
            e_fr = sbert_model.encode(b_fr, convert_to_numpy=True, show_progress_bar=False)
            e_lr = sbert_model.encode(b_lr, convert_to_numpy=True, show_progress_bar=False)
            for j, (v1, v2) in enumerate(zip(e_fr, e_lr)):
                sbert_scores[i+j] = float(max(0.0, cos_sim([v1], [v2])[0][0]))

    # Variables de surface
    rows = []
    for i, (fr, lr) in enumerate(zip(fr_sentences, lr_sentences)):
        rows.append([
            sbert_scores[i],
            length_score(fr, lr),
            jaccard_ngrams(fr, lr, 2),
            jaccard_ngrams(fr, lr, 3),
            jaccard_ngrams(fr, lr, 4),
            phonetic_jaccard(fr, lr),
        ])
    return np.array(rows)


print("Fonctions de calcul des variables définies.")

## 4. Chargement du corpus de validation et inférence

Le programme parcourt l'arborescence `validation/`, détecte les paires de fichiers `*_fr.txt` / `*_lr.txt` dans chaque sous-dossier `fr-XX/`, segmente les textes, puis applique le modèle sur **toutes les combinaisons** de phrases françaises et régionales au sein d'une même paire de documents (stratégie de corpus comparable non aligné).

In [ ]:
def discover_pairs(validation_dir: str) -> list:
    """
    Parcourt l'arborescence de validation et retourne une liste de tuples :
    (lang_code, pair_folder, fr_path, lr_path)

    Structure attendue :
    validation/
    └── <langue_code>/          (ex: alsacien_gsw)
        └── fr-<code>/          (ex: fr-gsw)
            ├── article_fr.txt
            └── article_gsw.txt (ou article_co.txt, article_oc.txt)
    """
    pairs = []
    if not os.path.isdir(validation_dir):
        print(f"[ERREUR] Dossier introuvable : {validation_dir}")
        return pairs

    for lang_folder in sorted(os.listdir(validation_dir)):
        lang_path = os.path.join(validation_dir, lang_folder)
        if not os.path.isdir(lang_path):
            continue
        lang_code = LANG_FOLDERS.get(lang_folder, lang_folder)

        for pair_folder in sorted(os.listdir(lang_path)):
            pair_path = os.path.join(lang_path, pair_folder)
            if not os.path.isdir(pair_path):
                continue

            # Identifie les fichiers _fr et les fichiers régionaux
            fr_files  = [f for f in os.listdir(pair_path) if f.endswith('_fr.txt')]
            lr_files  = [f for f in os.listdir(pair_path)
                         if f.endswith('.txt') and not f.endswith('_fr.txt')]

            # Appariement par préfixe commun
            for fr_file in fr_files:
                base = fr_file[:-7]  # supprime "_fr.txt"
                matched_lr = [f for f in lr_files if f.startswith(base + '_')]
                if not matched_lr:
                    # Si pas de correspondance par préfixe, prend le seul fichier lr disponible
                    matched_lr = lr_files
                for lr_file in matched_lr:
                    pairs.append((
                        lang_code,
                        f"{lang_folder}/{pair_folder}/{base}",
                        os.path.join(pair_path, fr_file),
                        os.path.join(pair_path, lr_file),
                    ))
    return pairs


pairs_found = discover_pairs(VALIDATION_DIR)
if pairs_found:
    print(f"{len(pairs_found)} paire(s) de fichiers détectée(s) :")
    for lang, name, fp_fr, fp_lr in pairs_found:
        print(f"  [{lang}] {name}")
        print(f"         FR : {fp_fr}")
        print(f"         LR : {fp_lr}")
else:
    print("Aucune paire trouvée. Vérifiez la structure du dossier validation/.")

In [ ]:
from tqdm.auto import tqdm

all_records = []

for lang_code, pair_name, fp_fr, fp_lr in tqdm(pairs_found, desc="Paires"):

    # --- Chargement et segmentation ---
    fr_sents = load_and_segment(fp_fr)
    lr_sents = load_and_segment(fp_lr)

    if not fr_sents or not lr_sents:
        print(f"  [AVERTISSEMENT] Aucune phrase extraite pour {pair_name} — ignoré.")
        continue

    print(f"\n[{lang_code}] {pair_name}")
    print(f"  FR : {len(fr_sents)} phrases | LR : {len(lr_sents)} phrases")
    print(f"  Comparaisons totales : {len(fr_sents) * len(lr_sents)}")

    # --- Génération de toutes les combinaisons ---
    fr_expanded = [fr for fr in fr_sents for _ in lr_sents]
    lr_expanded = [lr for _  in fr_sents for lr in lr_sents]

    # --- Calcul des variables ---
    sbert_arg = sbert_model if _sbert_available else None
    features  = compute_features_batch(fr_expanded, lr_expanded, sbert_model=sbert_arg)

    # --- Inférence ---
    features_sc   = scaler.transform(features)
    pred_bin      = model_bin.predict(features_sc)
    proba_bin     = model_bin.predict_proba(features_sc)[:, 1]   # P(parallèle)
    pred_mc       = model_mc.predict(features_sc)
    proba_mc_all  = model_mc.predict_proba(features_sc)
    conf_mc       = proba_mc_all[np.arange(len(pred_mc)), pred_mc]

    # Étiquette textuelle des classes multiclasse
    label_mc = [mc_classes[p] for p in pred_mc]

    # --- Construction du tableau de résultats ---
    for i in range(len(fr_expanded)):
        all_records.append({
            "langue":            lang_code,
            "paire":             pair_name,
            "fr_sentence":       fr_expanded[i],
            "lr_sentence":       lr_expanded[i],
            "pred_binaire":      int(pred_bin[i]),
            "confiance_bin":     round(float(proba_bin[i]), 4),
            "pred_classe":       label_mc[i],
            "confiance_mc":      round(float(conf_mc[i]), 4),
            "sbert_score":       round(float(features[i, 0]), 4),
            "length_score":      round(float(features[i, 1]), 4),
            "jaccard_bigrams":   round(float(features[i, 2]), 4),
            "jaccard_trigrams":  round(float(features[i, 3]), 4),
            "jaccard_quadrigrams": round(float(features[i, 4]), 4),
            "phonetic_jaccard":  round(float(features[i, 5]), 4),
        })

df_all = pd.DataFrame(all_records)
print(f"\nTotal comparaisons effectuées : {len(df_all)}")

## 5. Analyse des résultats

### 5.1 Vue d'ensemble des paires candidates

In [ ]:
df_candidates = df_all[
    (df_all["pred_binaire"] == 1) &
    (df_all["confiance_bin"] >= CONFIDENCE_THRESHOLD)
].copy().sort_values("confiance_bin", ascending=False)

print(f"Paires candidates retenues (seuil confiance ≥ {CONFIDENCE_THRESHOLD}) :")
print(f"  {len(df_candidates)} paires sur {len(df_all)} comparaisons ")
print(f"  ({100 * len(df_candidates) / max(len(df_all), 1):.1f}%)")
print()
print("Distribution par classe multiclasse :")
print(df_candidates["pred_classe"].value_counts().to_string())
print()
print("Distribution par langue :")
print(df_candidates["langue"].value_counts().to_string())

In [ ]:
# Aperçu des meilleures paires
print("=== Top 10 paires candidates (par confiance décroissante) ===")
for _, row in df_candidates.head(10).iterrows():
    print(f"\n[{row['langue']}] classe={row['pred_classe']} | confiance={row['confiance_bin']:.2%} | SBERT={row['sbert_score']:.3f}")
    print(f"  FR : {row['fr_sentence'][:100]}")
    print(f"  LR : {row['lr_sentence'][:100]}")

### 5.2 Comparaison des performances par langue

In [ ]:
lang_stats = []
for lang in df_all["langue"].unique():
    sub      = df_all[df_all["langue"] == lang]
    cand     = sub[(sub["pred_binaire"] == 1) & (sub["confiance_bin"] >= CONFIDENCE_THRESHOLD)]
    parallel = cand[cand["pred_classe"] == "parallel"]
    semi     = cand[cand["pred_classe"] == "semi-parallel"]
    lang_stats.append({
        "Langue":              lang,
        "Comparaisons totales": len(sub),
        "Paires candidates":   len(cand),
        "dont parallèles":     len(parallel),
        "dont semi-parallèles": len(semi),
        "Taux extraction (%)": round(100 * len(cand) / max(len(sub), 1), 2),
        "Confiance moy. (bin)": round(cand["confiance_bin"].mean(), 4) if len(cand) else 0,
        "SBERT moy. (candidats)": round(cand["sbert_score"].mean(), 4) if len(cand) else 0,
    })

lang_df = pd.DataFrame(lang_stats)
print("=== Statistiques par langue ===")
print(lang_df.to_string(index=False))

### 5.3 Visualisations

In [ ]:
langs = df_all["langue"].unique()
n_langs = len(langs)

if n_langs == 0:
    print("Aucune donnée à visualiser.")
else:
    fig, axes = plt.subplots(1, max(n_langs, 1), figsize=(6 * max(n_langs, 1), 5), sharey=False)
    if n_langs == 1:
        axes = [axes]

    for ax, lang in zip(axes, langs):
        sub = df_all[df_all["langue"] == lang]["sbert_score"]
        sub_cand = df_candidates[df_candidates["langue"] == lang]["sbert_score"]
        ax.hist(sub,      bins=30, alpha=0.5, color="steelblue", label="toutes comparaisons")
        ax.hist(sub_cand, bins=30, alpha=0.7, color="#2ecc71",   label="candidates retenues")
        ax.axvline(sub_cand.mean() if len(sub_cand) else 0,
                   color="darkgreen", linestyle="--", linewidth=1.5,
                   label=f"moy. candidates = {sub_cand.mean():.2f}" if len(sub_cand) else "")
        ax.set_title(f"SBERT — {lang}")
        ax.set_xlabel("Score SBERT")
        ax.set_ylabel("Fréquence")
        ax.legend(fontsize=8)

    plt.suptitle("Distribution des scores SBERT par langue\n(toutes comparaisons vs paires candidates)",
                 fontsize=11)
    plt.tight_layout()
    plt.savefig("validation_sbert_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : validation_sbert_distribution.png")

In [ ]:
if len(df_candidates) > 0:
    # Barplot : nombre de paires candidates par langue et par classe
    pivot = df_candidates.groupby(["langue", "pred_classe"]).size().unstack(fill_value=0)

    colors_cls = {"parallel": "#2ecc71", "semi-parallel": "#f39c12", "non-parallel": "#e74c3c"}
    col_order  = [c for c in ["parallel", "semi-parallel", "non-parallel"] if c in pivot.columns]
    pivot[col_order].plot(
        kind="bar",
        color=[colors_cls.get(c, "gray") for c in col_order],
        figsize=(8, 4),
        edgecolor="white",
    )
    plt.title("Paires candidates par langue et par classe")
    plt.xlabel("Langue")
    plt.ylabel("Nombre de paires")
    plt.xticks(rotation=0)
    plt.legend(title="Classe")
    plt.tight_layout()
    plt.savefig("validation_candidates_by_lang.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : validation_candidates_by_lang.png")

In [ ]:
if len(df_candidates) > 0:
    # Heatmap : score SBERT moyen par paire de documents et par classe
    pivot_heat = df_candidates.groupby(["paire", "pred_classe"])["sbert_score"].mean().unstack(fill_value=0)
    if not pivot_heat.empty:
        fig, ax = plt.subplots(figsize=(8, max(3, len(pivot_heat) * 0.5)))
        sns.heatmap(pivot_heat, annot=True, fmt=".2f", cmap="YlGn", ax=ax,
                    linewidths=0.5, vmin=0, vmax=1)
        ax.set_title("Score SBERT moyen par paire de documents et classe")
        ax.set_xlabel("Classe")
        ax.set_ylabel("Paire de documents")
        plt.tight_layout()
        plt.savefig("validation_sbert_heatmap.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Figure sauvegardée : validation_sbert_heatmap.png")

### 5.4 Effet du seuil de confiance

Cette cellule permet d'observer comment le nombre de paires candidates et la confiance moyenne évoluent selon le seuil choisi, afin d'orienter le réglage du paramètre `CONFIDENCE_THRESHOLD`.

In [ ]:
thresholds  = np.arange(0.5, 0.96, 0.05)
n_candidates = []
mean_sbert   = []

for thr in thresholds:
    cand = df_all[(df_all["pred_binaire"] == 1) & (df_all["confiance_bin"] >= thr)]
    n_candidates.append(len(cand))
    mean_sbert.append(cand["sbert_score"].mean() if len(cand) else 0)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.bar(thresholds, n_candidates, width=0.04, alpha=0.6,
        color="steelblue", label="Nombre de paires candidates")
ax2.plot(thresholds, mean_sbert, 'o-', color="darkorange",
         linewidth=2, label="SBERT moyen des candidates")

ax1.axvline(CONFIDENCE_THRESHOLD, color="red", linestyle="--",
            linewidth=1.5, label=f"Seuil actuel = {CONFIDENCE_THRESHOLD}")
ax1.set_xlabel("Seuil de confiance")
ax1.set_ylabel("Nombre de paires candidates", color="steelblue")
ax2.set_ylabel("Score SBERT moyen", color="darkorange")
ax1.set_title("Effet du seuil de confiance sur les paires candidates")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("validation_threshold_effect.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : validation_threshold_effect.png")

## 6. Export CSV des paires candidates

In [ ]:
# Export complet des paires candidates triées par confiance décroissante
df_candidates_sorted = df_candidates.sort_values(
    ["langue", "confiance_bin"], ascending=[True, False]
)

df_candidates_sorted.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Export CSV : {OUTPUT_CSV}")
print(f"  {len(df_candidates_sorted)} paires candidates exportées.")
print(f"\nAperçu des colonnes :")
print(df_candidates_sorted.columns.tolist())
df_candidates_sorted.head(5)

In [ ]:
# Export séparé par langue pour faciliter l'analyse manuelle
for lang in df_candidates_sorted["langue"].unique():
    sub = df_candidates_sorted[df_candidates_sorted["langue"] == lang]
    outfile = f"validation_candidates_{lang}.csv"
    sub.to_csv(outfile, index=False, encoding="utf-8-sig")
    print(f"  ✔ {outfile} — {len(sub)} paires")

## 7. Résumé final

In [ ]:
print("=" * 60)
print("RÉSUMÉ DE LA VALIDATION")
print("=" * 60)
print(f"Dossier de validation     : {VALIDATION_DIR}")
print(f"Paires de fichiers lues   : {len(pairs_found)}")
print(f"Comparaisons effectuées   : {len(df_all)}")
print(f"Seuil de confiance        : {CONFIDENCE_THRESHOLD}")
print(f"Paires candidates retenues: {len(df_candidates)}  "
      f"({100*len(df_candidates)/max(len(df_all),1):.1f}%)")
print()
print(lang_df.to_string(index=False))
print()
print("Fichiers exportés :")
print(f"  {OUTPUT_CSV}")
for lang in df_candidates_sorted["langue"].unique() if len(df_candidates) else []:
    print(f"  validation_candidates_{lang}.csv")
print("Figures exportées :")
for fig_name in ["validation_sbert_distribution.png",
                  "validation_candidates_by_lang.png",
                  "validation_sbert_heatmap.png",
                  "validation_threshold_effect.png"]:
    print(f"  {fig_name}")